# Module 1: Data Ingestion + SQL Layer


**Goal:** Load IPL CSVs → clean → push to SQLite → run  SQL queries



## Step 1 — Load & Inspect Data

In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import os

# Load raw data
matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

print('=== SHAPE ===')
print('matches:', matches.shape)
print('deliveries:', deliveries.shape)

print('\n=== MATCHES COLUMNS ===')
print(matches.columns.tolist())

print('\n=== DELIVERIES COLUMNS ===')
print(deliveries.columns.tolist())

=== SHAPE ===
matches: (1095, 20)
deliveries: (260920, 17)

=== MATCHES COLUMNS ===
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

=== DELIVERIES COLUMNS ===
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


In [2]:
print('=== MATCHES NULL COUNTS ===')
print(matches.isnull().sum()[matches.isnull().sum() > 0])

print('\n=== DELIVERIES NULL COUNTS ===')
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0])

print('\n=== MATCHES SAMPLE ===')
matches.head(3)

=== MATCHES NULL COUNTS ===
city                 51
player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64

=== DELIVERIES NULL COUNTS ===
extras_type         246795
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

=== MATCHES SAMPLE ===


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar


## Step 2 — Cleaning Data

In [4]:
# Fix column names: lowercase + underscores
matches.columns = matches.columns.str.lower().str.replace(' ', '_')
deliveries.columns = deliveries.columns.str.lower().str.replace(' ', '_')

# Fill nulls in matches
matches['city'] = matches['city'].fillna('Unknown')
matches['winner'] = matches['winner'].fillna('No Result')
matches['player_of_match'] = matches['player_of_match'].fillna('N/A')

# Remove super overs from deliveries
if 'is_super_over' in deliveries.columns:
    deliveries = deliveries[deliveries['is_super_over'] == 0]
    print('Super overs removed.')

# Ensure match_id column name is consistent
if 'id' in matches.columns and 'match_id' not in matches.columns:
    matches.rename(columns={'id': 'match_id'}, inplace=True)

if 'id' in deliveries.columns and 'match_id' not in deliveries.columns:
    deliveries.rename(columns={'id': 'match_id'}, inplace=True)

print('matches shape after cleaning:', matches.shape)
print('deliveries shape after cleaning:', deliveries.shape)

print('\nMatches columns:', matches.columns.tolist())
matches.head(2)

matches shape after cleaning: (1095, 20)
deliveries shape after cleaning: (260920, 17)

Matches columns: ['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']


,match_id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri


In [5]:
# Saving cleaned files
os.makedirs('../data/processed', exist_ok=True)
matches.to_csv('../data/processed/matches_clean.csv', index=False)
deliveries.to_csv('../data/processed/deliveries_clean.csv', index=False)
print('Cleaned CSVs saved to data/processed/')

Cleaned CSVs saved to data/processed/


## Step 3 — Pushing to SQLite Database

In [6]:
# Creating SQLite DB one level above notebooks/
engine = create_engine('sqlite:///../ipl.db')

matches.to_sql('matches', engine, if_exists='replace', index=False)
deliveries.to_sql('deliveries', engine, if_exists='replace', index=False)

print('Database created: ipl.db')
print('Tables: matches, deliveries')

# Verifying tables in DB
with engine.connect() as conn:
    from sqlalchemy import text
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
    print('Tables in DB:', [r[0] for r in result])

Database created: ipl.db
Tables: matches, deliveries
Tables in DB: ['matches', 'deliveries']


## Step 4 — SQL Queries (5 Key Queries)

All queries run via `pd.read_sql()`

In [8]:
deliveries.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs',
       'total_runs', 'extras_type', 'is_wicket', 'player_dismissed',
       'dismissal_kind', 'fielder'],
      dtype='object')

In [9]:
# --- QUERY 1: Top 10 Run Scorers All Time ---
query1 = '''
SELECT 
    batter,
    SUM(batsman_runs) AS total_runs,
    COUNT(*) AS balls_faced,
    ROUND(SUM(batsman_runs) * 100.0 / COUNT(*), 2) AS strike_rate
FROM deliveries
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 10
'''
top_batsmen = pd.read_sql(query1, engine)
print('=== TOP 10 RUN SCORERS ===')
top_batsmen

=== TOP 10 RUN SCORERS ===


,batter,total_runs,balls_faced,strike_rate
0,V Kohli,8014,6236,128.51
1,S Dhawan,6769,5483,123.45
2,RG Sharma,6630,5183,127.92
3,DA Warner,6567,4849,135.43
4,SK Raina,5536,4177,132.54
5,MS Dhoni,5243,3947,132.84
6,AB de Villiers,5181,3487,148.58
7,CH Gayle,4997,3516,142.12
8,RV Uthappa,4954,3927,126.15
9,KD Karthik,4843,3687,131.35


In [10]:
# --- QUERY 2: Team Win Percentage ---
query2 = '''
SELECT 
    team1 AS team,
    COUNT(*) AS total_matches,
    SUM(CASE WHEN winner = team1 THEN 1 ELSE 0 END) AS wins,
    ROUND(SUM(CASE WHEN winner = team1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS win_pct
FROM matches
WHERE winner != 'No Result'
GROUP BY team1
ORDER BY win_pct DESC
'''
team_win = pd.read_sql(query2, engine)
print('=== TEAM WIN PERCENTAGE ===')
team_win

=== TEAM WIN PERCENTAGE ===


,team,total_matches,wins,win_pct
0,Lucknow Super Giants,22,16,72.73
1,Rising Pune Supergiant,7,5,71.43
2,Chennai Super Kings,128,75,58.59
3,Mumbai Indians,123,70,56.91
4,Delhi Capitals,41,23,56.10
5,Rajasthan Royals,101,55,54.46
6,Kolkata Knight Riders,121,65,53.72
7,Gujarat Titans,21,11,52.38
8,Sunrisers Hyderabad,86,44,51.16
9,Royal Challengers Bangalore,132,66,50.00


In [11]:
# --- QUERY 3: Best Bowlers by Economy Rate (min 50 overs) ---
query3 = '''
SELECT 
    bowler,
    COUNT(DISTINCT match_id) AS matches,
    SUM(total_runs) AS runs_given,
    COUNT(*) / 6 AS overs,
    ROUND(SUM(total_runs) * 6.0 / COUNT(*), 2) AS economy
FROM deliveries
GROUP BY bowler
HAVING overs >= 50
ORDER BY economy ASC
LIMIT 10
'''
best_bowlers = pd.read_sql(query3, engine)
print('=== BEST BOWLERS BY ECONOMY (min 50 overs) ===')
best_bowlers

=== BEST BOWLERS BY ECONOMY (min 50 overs) ===


,bowler,matches,runs_given,overs,economy
0,A Kumble,42,1089,163,6.65
1,GD McGrath,14,366,54,6.67
2,M Muralitharan,66,1765,263,6.70
3,J Yadav,20,447,66,6.74
4,SP Narine,175,4672,691,6.76
5,DW Steyn,95,2583,380,6.79
6,RE van der Merwe,21,515,75,6.79
7,DL Vettori,34,894,130,6.83
8,Rashid Khan,121,3340,483,6.91
9,J Botha,34,818,118,6.92


In [12]:
# --- QUERY 4: Toss Win vs Match Win Correlation ---
query4 = '''
SELECT 
    toss_decision,
    COUNT(*) AS total,
    SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS toss_win_match_win,
    ROUND(SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct
FROM matches
WHERE winner != 'No Result'
GROUP BY toss_decision
'''
toss_analysis = pd.read_sql(query4, engine)
print('=== TOSS WIN vs MATCH WIN CORRELATION ===')
toss_analysis

=== TOSS WIN vs MATCH WIN CORRELATION ===


,toss_decision,total,toss_win_match_win,pct
0,bat,390,177,45.38
1,field,700,377,53.86


In [13]:
# --- QUERY 5: Phase-wise Run Rate (Powerplay / Middle / Death) ---
query5 = '''
SELECT
    CASE 
        WHEN over BETWEEN 0 AND 5 THEN 'Powerplay (1-6)'
        WHEN over BETWEEN 6 AND 14 THEN 'Middle (7-15)'
        ELSE 'Death (16-20)'
    END AS phase,
    ROUND(SUM(total_runs) * 6.0 / COUNT(*), 2) AS run_rate,
    COUNT(*) AS total_balls,
    SUM(total_runs) AS total_runs
FROM deliveries
GROUP BY phase
'''
phase_rr = pd.read_sql(query5, engine)
print('=== PHASE-WISE RUN RATE ===')
phase_rr

=== PHASE-WISE RUN RATE ===


,phase,run_rate,total_balls,total_runs
0,Death (16-20),9.47,59463,93884
1,Middle (7-15),7.56,119552,150655
2,Powerplay (1-6),7.56,81905,103217


In [14]:
# Saving phase_rr for use in Module 2
phase_rr.to_csv('../data/processed/phase_run_rate.csv', index=False)
team_win.to_csv('../data/processed/team_win_pct.csv', index=False)
top_batsmen.to_csv('../data/processed/top_batsmen_sql.csv', index=False)

print('\n✅ MODULE 1 COMPLETE!')
print('ipl.db created ✓')
print('5 SQL queries executed ✓')
print('Cleaned CSVs saved to data/processed/ ✓')


✅ MODULE 1 COMPLETE!
ipl.db created ✓
5 SQL queries executed ✓
Cleaned CSVs saved to data/processed/ ✓


**Outcome:** `ipl.db` created in root, all 5 queries return results, cleaned CSVs saved to `data/processed/`